# Day 3 — Hybrid Search: BM25 + Dense + RRF

Neither keyword search nor semantic search alone beats the combination.
BM25 wins on exact terms ('AMOLED 120Hz'). Dense wins on meaning ('lightweight for travel').
RRF fuses both without requiring scores to be on the same scale — making hybrid search the industry standard for production RAG.

In [ ]:
import sys
sys.path.insert(0, '../src')
import numpy as np
from day3.hybrid_search import (
    BM25Index, DenseIndex, HybridSearcher,
    reciprocal_rank_fusion, generate_product_corpus, SearchResult
)

# Mock embeddings for demo (no model download required)
def mock_embed(texts):
    vecs = []
    for t in texts:
        rng = np.random.default_rng(abs(hash(t)) % (2**31))
        v = rng.standard_normal(384).astype(np.float32)
        v = v / np.linalg.norm(v)
        vecs.append(v)
    return np.array(vecs)

corpus = generate_product_corpus()
print(f"Product corpus: {len(corpus)} documents")
print(f"Sample: '{corpus[0][:80]}...'")

## 1. BM25 Keyword Search

BM25 (Best Match 25) is the gold standard for keyword retrieval.
It improves on TF-IDF with term frequency saturation (repeated keywords give diminishing returns) and document length normalisation.

In [ ]:
bm25 = BM25Index()
bm25.index(corpus)

query = "AMOLED 120Hz display smartphone"
results = bm25.search(query, n=5)
print(f"BM25 results for: '{query}'")
for r in results:
    print(f"  [{r.rank}] score={r.score:.3f} | {r.text[:90]}")

## 2. Dense Semantic Search

Dense retrieval embeds all documents and the query into the same vector space, then finds nearest neighbours by cosine similarity.
It understands meaning, not just word matching.

In [ ]:
dense = DenseIndex(mock_embed_fn=mock_embed)
dense.index(corpus)

semantic_query = "best laptop for machine learning and data science"
results = dense.search(semantic_query, n=5)
print(f"Dense results for: '{semantic_query}'")
for r in results:
    print(f"  [{r.rank}] score={r.score:.4f} | {r.text[:90]}")

## 3. Reciprocal Rank Fusion (RRF)

RRF merges ranked lists without requiring scores to be on the same scale.
Formula: score(d) = Σ 1/(k + rank_i(d)) where k=60.
Documents appearing in multiple lists get higher fused scores.

In [ ]:
# Manual RRF demo
list_a = [
    SearchResult("MacBook Pro 14 M3 Pro", 10.0, 1, "bm25"),
    SearchResult("Dell XPS 15 OLED", 8.0, 2, "bm25"),
    SearchResult("ThinkPad X1 Carbon", 5.0, 3, "bm25"),
]
list_b = [
    SearchResult("ThinkPad X1 Carbon", 0.95, 1, "dense"),
    SearchResult("MacBook Pro 14 M3 Pro", 0.90, 2, "dense"),
    SearchResult("ASUS ROG Zephyrus G14", 0.85, 3, "dense"),
]

fused = reciprocal_rank_fusion([list_a, list_b])
print("BM25 ranks:   MacBook(#1) > Dell(#2) > ThinkPad(#3)")
print("Dense ranks:  ThinkPad(#1) > MacBook(#2) > ROG(#3)")
print("\nRRF Fusion:")
for r in fused:
    print(f"  [{r.rank}] {r.text:<30} RRF_score={r.score:.6f}")
print("\nMacBook appears in both lists \u2192 highest fused score")
print("k=60 formula: 1/(60+1) + 1/(60+2) = 0.01639 + 0.01613 = 0.03252")

## 4. Hybrid Searcher

The HybridSearcher runs BM25 and dense in parallel, then fuses results with RRF.
Three demonstration queries show when each method wins.

In [ ]:
searcher = HybridSearcher(bm25_weight=0.3, dense_weight=0.7, mock_embed_fn=mock_embed)
searcher.index(corpus)

test_queries = [
    ("AMOLED 120Hz display",           "BM25 wins \u2014 exact spec terms"),
    ("best laptop for ML",              "Dense wins \u2014 semantic meaning"),
    ("lightweight device for travel",   "Hybrid wins \u2014 both signals"),
]

for query, explanation in test_queries:
    comparison = searcher.compare(query, n=3)
    print(f"\nQuery: '{query}'  ({explanation})")
    print(f"  BM25  top-1: {comparison['bm25_results'][0].text[:80]}")
    print(f"  Dense top-1: {comparison['dense_results'][0].text[:80]}")
    print(f"  Hybrid top-1: {comparison['hybrid_results'][0].text[:80]}")

## 5. BM25 vs Dense vs Hybrid — Detailed Comparison

Run a full side-by-side comparison showing why hybrid search wins overall across different query types.

In [ ]:
print(f"{'Query Type':<35} {'BM25':>6} {'Dense':>6} {'Hybrid':>8}")
print("-" * 60)

queries_info = [
    ("AMOLED 120Hz smartphone",  "exact"),
    ("machine learning laptop",  "semantic"),
    ("noise cancelling travel",  "hybrid"),
    ("Snapdragon 8 Gen 3",       "exact"),
    ("portable lightweight",     "semantic"),
]

for query, qtype in queries_info:
    bm25_r = bm25.search(query, n=1)
    dense_r = dense.search(query, n=1)
    hybrid_r = searcher.search(query, n=1)
    
    b_score = bm25_r[0].score if bm25_r else 0
    d_score = dense_r[0].score if dense_r else 0
    h_score = hybrid_r[0].score if hybrid_r else 0
    
    print(f"{query:<35} {b_score:>6.3f} {d_score:>6.4f} {h_score:>8.5f}  ({qtype})")

## Databricks Bridge

Databricks Vector Search supports hybrid search natively.
You configure both the embedding model and BM25 when creating the index.

In [ ]:
print("""
DATABRICKS VECTOR SEARCH \u2014 HYBRID MODE:

  from databricks.vector_search.client import VectorSearchClient
  
  vsc = VectorSearchClient()
  index = vsc.create_delta_sync_index(
      endpoint_name     = "one-env-shared-endpoint-0",
      source_table      = "dev_agents.naval.product_chunks",
      index_name        = "dev_agents.naval.foodly_index",
      primary_key       = "chunk_id",
      embedding_source_column = "text",
      embedding_model_endpoint_name = "databricks-bge-large-en",
  )
  
  # Hybrid search: keyword + semantic
  results = index.similarity_search(
      query_text = "noise cancelling headphones for travel",
      columns    = ["chunk_id", "text", "category"],
      num_results = 10,
      filters    = {"category": "headphones"},
      query_type = "HYBRID"  # \u2190 enables BM25 + dense fusion
  )
""")